##### Imports:

In [2]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm


In [19]:
# Load matched dataset
df = pd.read_parquet("Data/lr_epc_matched.parquet")

# Load postcode directory — only columns we need
postcodes = pd.read_csv(
    "Data/Online_ONS_Postcode_Directory_Live.csv",
    usecols=["PCDS", "LAT", "LONG"]
)

# Normalise postcode format for merging
df["postcode_merge"] = df["postcode_x"].str.strip().str.upper()
postcodes["PCDS"] = postcodes["PCDS"].str.strip().str.upper()

# Merge
df = df.merge(postcodes, left_on="postcode_merge", right_on="PCDS", how="left")

# Check coverage
matched_geo = df["LAT"].notna().sum()
print(f"Properties with lat/lon: {matched_geo:,} ({matched_geo/len(df)*100:.1f}%)")
print(f"Missing lat/lon:         {df['LAT'].isna().sum():,}")

df.head(2)

Properties with lat/lon: 5,770,097 (99.6%)
Missing lat/lon:         23,099


,transaction_id,price,date,postcode_x,property_type_x,new_build,duration,paon,saon,street,...,property_type_y,transaction_type,construction_age_band,built_form,main_fuel,mains_gas_flag,postcode_merge,PCDS,LAT,LONG
0,{01EB45EF-612C-40F3-E063-4704A8C05FDE},1281150,2023-06-16,N7 7BG,T,N,F,31,NaN,BRYANTWOOD ROAD,...,House,Marketed sale,England and Wales: 1900-1929,Mid-Terrace,mains gas (not community),Y,N7 7BG,N7 7BG,51.55244,-0.107227
1,{01EB45EF-612D-40F3-E063-4704A8C05FDE},1675000,2023-06-23,SE22 8UG,T,N,F,33,NaN,BEAUVAL ROAD,...,House,Marketed sale,England and Wales: before 1900,Mid-Terrace,mains gas (not community),Y,SE22 8UG,SE22 8UG,51.45220,-0.078014


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Load
df = pd.read_parquet("Data/lr_epc_matched.parquet")
postcodes = pd.read_csv(
    "Data/Online_ONS_Postcode_Directory_Live.csv",
    usecols=["PCDS", "LAT", "LONG"]
)
postcodes["PCDS"] = postcodes["PCDS"].str.strip().str.upper()
df["postcode_merge"] = df["postcode_x"].str.strip().str.upper()
df = df.merge(postcodes, left_on="postcode_merge", right_on="PCDS", how="left")

# Clean
df = df[df["LAT"].notna() & (df["LAT"] != 99.999999)]
df["has_epc"] = df["lodgement_date"].notna()

# Sample
matched   = df[df["has_epc"]]
unmatched = df[~df["has_epc"]]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 16), sharey=True)

for ax, data, color, title in [
    (axes[0], matched,   "#1D9E75", "Matched to EPC"),
    (axes[1], unmatched, "#E24B4A", "No EPC match"),
]:
    ax.scatter(data["LONG"], data["LAT"], s=0.3, alpha=0.3, color=color, linewidths=0)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Longitude")
    ax.set_aspect("equal")
    ax.axis("off")

axes[0].set_ylabel("Latitude")
fig.suptitle("Property Sales — EPC Match Coverage", fontsize=16, y=0.92)
plt.tight_layout()
plt.savefig("epc_coverage_map.png", dpi=150, bbox_inches="tight")
plt.show()

ValueError: Cannot take a larger sample than population when 'replace=False'